1. Imports

In [ ]:
import pyarrow.dataset as ds
import pyarrow.compute as pc
import numpy as np
import matplotlib.pyplot as plt
import time

2. Configuração

In [ ]:
DATA_PATH = "data/Indian_Weather_Dataset.parquet"

BATCH_SIZE = 8192
EPOCHS = 3
LR = 0.001

TARGET = "rain_label"

CAT_COLS = ["state", "city", "crops"]

NUM_COLS = [
    "lat","lon","temperature_C","humidity_pct","pressure_hPa",
    "dew_point_C","pressure_trend","solar_radiation_Wm2",
    "wind_speed_ms","cloud_cover_pct","hour","month",
    "wind_dir_sin","wind_dir_cos","et0_mm"
]

3. Ativações e loss

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -40, 40)))

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(np.float32)

def bce(y, yhat):
    eps = 1e-7
    yhat = np.clip(yhat, eps, 1 - eps)
    return -np.mean(y*np.log(yhat) + (1-y)*np.log(1-yhat))

4. Modelo

In [ ]:
class MLP:
    def __init__(self, input_dim):
        self.W1 = np.random.randn(input_dim, 64)*0.01
        self.b1 = np.zeros((1,64))

        self.W2 = np.random.randn(64,32)*0.01
        self.b2 = np.zeros((1,32))

        self.W3 = np.random.randn(32,1)*0.01
        self.b3 = np.zeros((1,1))

    def forward(self,X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = relu(self.z1)

        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = relu(self.z2)

        self.z3 = self.a2 @ self.W3 + self.b3
        self.out = sigmoid(self.z3)
        return self.out

    def backward(self,X,y,lr):
        n = X.shape[0]

        dz3 = (self.out - y)/n
        dW3 = self.a2.T @ dz3

        dz2 = (dz3 @ self.W3.T)*relu_grad(self.z2)
        dW2 = self.a1.T @ dz2

        dz1 = (dz2 @ self.W2.T)*relu_grad(self.z1)
        dW1 = X.T @ dz1

        self.W3 -= lr*dW3
        self.W2 -= lr*dW2
        self.W1 -= lr*dW1

5. Dataset

In [ ]:
dataset = ds.dataset(DATA_PATH, format="parquet")

train_end = np.datetime64("2017-02-23T11:00:00")
val_end   = np.datetime64("2021-09-05T23:00:00")

train_filter = pc.field("datetime") < train_end
val_filter   = (pc.field("datetime") >= train_end) & (pc.field("datetime") < val_end)
test_filter  = pc.field("datetime") >= val_end

6. Categorias (one-hot simples)

In [ ]:
def build_maps():
    maps = {}
    for col in CAT_COLS:
        vals = dataset.to_table(columns=[col])[col].to_pylist()
        maps[col] = {v:i for i,v in enumerate(set(vals))}
    return maps

cat_maps = build_maps()

7. Normalização (treino)

In [ ]:
def get_stats():
    table = dataset.to_table(columns=NUM_COLS, filter=train_filter)
    means = []
    stds = []

    for col in NUM_COLS:
        arr = np.array(table[col])
        means.append(arr.mean())
        std = arr.std()
        stds.append(std if std>0 else 1)

    return np.array(means), np.array(stds)

means, stds = get_stats()

8. Batch → NumPy

In [ ]:
def to_xy(batch):
    d = batch.to_pydict()

    Xn = np.stack([np.array(d[c]) for c in NUM_COLS], axis=1)
    Xn = (Xn - means)/stds

    cats = []
    for c in CAT_COLS:
        idx = np.array([cat_maps[c].get(v,0) for v in d[c]])
        oh = np.zeros((len(idx), len(cat_maps[c])))
        oh[np.arange(len(idx)), idx] = 1
        cats.append(oh)

    X = np.concatenate([Xn] + cats, axis=1)
    y = np.array(d[TARGET]).reshape(-1,1)

    return X.astype(np.float32), y.astype(np.float32)

9. Treino

In [ ]:
input_dim = len(NUM_COLS) + sum(len(cat_maps[c]) for c in CAT_COLS)
model = MLP(input_dim)

train_loss_hist = []
val_loss_hist = []

start = time.time()

for epoch in range(EPOCHS):
    losses = []

    scanner = dataset.scanner(
        columns=NUM_COLS+CAT_COLS+[TARGET],
        filter=train_filter,
        batch_size=BATCH_SIZE
    )

    for batch in scanner.to_batches():
        X,y = to_xy(batch)

        pred = model.forward(X)
        loss = bce(y,pred)

        model.backward(X,y,LR)
        losses.append(loss)

    train_loss = np.mean(losses)
    train_loss_hist.append(train_loss)

    print(f"Epoch {epoch+1} | loss={train_loss:.4f}")

10. Avaliação

In [ ]:
def evaluate(filt):
    scanner = dataset.scanner(
        columns=NUM_COLS+CAT_COLS+[TARGET],
        filter=filt,
        batch_size=BATCH_SIZE
    )

    tp=fp=tn=fn=0

    for batch in scanner.to_batches():
        X,y = to_xy(batch)
        p = model.forward(X) >= 0.5

        tp += np.sum((p==1)&(y==1))
        tn += np.sum((p==0)&(y==0))
        fp += np.sum((p==1)&(y==0))
        fn += np.sum((p==0)&(y==1))

    acc = (tp+tn)/(tp+tn+fp+fn)
    prec = tp/(tp+fp+1e-9)
    rec = tp/(tp+fn+1e-9)
    f1 = 2*prec*rec/(prec+rec+1e-9)

    return acc,prec,rec,f1

print("VAL:", evaluate(val_filter))
print("TEST:", evaluate(test_filter))

11. Gráfico

In [ ]:
plt.plot(train_loss_hist)
plt.title("Loss")
plt.show()